# Transformação e preparação de dados com Python

Projeto de portfólio focado em **exploração, limpeza, transformação e preparação de dados para Machine Learning**.

## Objetivos
- explorar dados numéricos e categóricos;
- tratar valores ausentes;
- identificar valores extremos;
- criar novas variáveis (*feature engineering*);
- analisar correlações;
- aplicar transformações logarítmicas e normalização Min-Max;
- observar o impacto das transformações em um modelo de regressão linear.

**Tecnologias:** Python, Pandas, NumPy, Matplotlib, Seaborn e Scikit-learn.

## 1. Bibliotecas

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LinearRegression

pd.set_option("display.max_columns", 100)
pd.set_option("display.precision", 2)

## 2. Titanic — exploração e limpeza

O conjunto Titanic é usado para demonstrar operações comuns de preparação de dados. Para facilitar a reprodução, o notebook tenta usar o arquivo local `data/titanic/train.csv`.

Colunas relevantes incluem `Survived`, `Pclass`, `Sex`, `Age`, `SibSp`, `Parch`, `Fare`, `Cabin` e `Embarked`.

In [ ]:
from pathlib import Path

TITANIC_PATH = Path("data/titanic/train.csv")
if not TITANIC_PATH.exists():
    raise FileNotFoundError(
        "Adicione o arquivo Titanic train.csv em data/titanic/train.csv. "
        "Veja o README para a origem dos dados."
    )

titanic = pd.read_csv(TITANIC_PATH)
print("Dimensões:", titanic.shape)
titanic.head()

In [ ]:
print(titanic.describe(include="all").T)

### Remoção de atributos pouco úteis

`PassengerId` é um identificador e `Ticket` é removido nesta etapa para manter o foco nas variáveis utilizadas no exemplo original. `Name` e `Cabin` podem ser explorados em trabalhos futuros.

In [ ]:
titanic = titanic.drop(columns=["PassengerId", "Ticket"], errors="ignore")
print(titanic.shape)

### Transformação de variáveis categóricas

A representação textual pode facilitar a análise exploratória. O rótulo original `Survived` é preservado para não alterar a variável-alvo.

In [ ]:
if "Survived" in titanic.columns:
    titanic["Survived_label"] = titanic["Survived"].map({0: "Não sobreviveu", 1: "Sobreviveu"})

if "Pclass" in titanic.columns:
    titanic["Pclass_label"] = titanic["Pclass"].map({1: "1ª classe", 2: "2ª classe", 3: "3ª classe"})

titanic[[c for c in ["Survived", "Survived_label", "Pclass", "Pclass_label"] if c in titanic.columns]].head()

### Engenharia de atributos: cabine e família

In [ ]:
if "Cabin" in titanic.columns:
    titanic["CabinDeck"] = titanic["Cabin"].fillna("Unknown").astype(str).str[0]

if {"SibSp", "Parch"}.issubset(titanic.columns):
    titanic["FamilySize"] = titanic["SibSp"] + titanic["Parch"]

titanic.head()

### Valores ausentes

Para `Age`, a mediana é usada como exemplo de imputação robusta a valores extremos.

In [ ]:
if "Age" in titanic.columns:
    print("Valores ausentes em Age antes:", titanic["Age"].isna().sum())
    titanic["Age"] = titanic["Age"].fillna(titanic["Age"].median())
    print("Valores ausentes em Age depois:", titanic["Age"].isna().sum())

    titanic["Age"].hist(bins=20, figsize=(8, 5))
    plt.title("Distribuição de idade após imputação")
    plt.xlabel("Idade")
    plt.ylabel("Frequência")
    plt.show()

### Detecção visual de outliers

In [ ]:
if "Fare" in titanic.columns:
    plt.figure(figsize=(8, 5))
    sns.boxplot(x=titanic["Fare"])
    plt.title("Distribuição de Fare")
    plt.show()

### Correlação entre variáveis numéricas

In [ ]:
numeric = titanic.select_dtypes(include=np.number)
plt.figure(figsize=(9, 7))
sns.heatmap(numeric.corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Matriz de correlação — Titanic")
plt.show()

## 3. Transformação numérica em uma base de imóveis

A segunda parte preserva o exemplo de regressão sobre a base de imóveis utilizada no material original. O arquivo deve ser colocado em `data/gazola/Gazola_dados_apartamento_resumo.xls`.

A análise compara dados originais com uma versão transformada por logaritmo e normalização Min-Max.

In [ ]:
GAZOLA_PATH = Path("data/gazola/Gazola_dados_apartamento_resumo.xls")
if GAZOLA_PATH.exists():
    gazola = pd.read_excel(GAZOLA_PATH, sheet_name=1)
    print("Dimensões:", gazola.shape)
    display(gazola.head())
else:
    print("Arquivo Gazola não encontrado. Coloque-o em data/gazola/ para executar esta seção.")

In [ ]:
if GAZOLA_PATH.exists():
    X = gazola.iloc[:, 1:-2].copy()
    y = gazola.iloc[:, -1].copy()
    atributos = list(X.columns)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.1, random_state=0
    )
    model = LinearRegression().fit(X_train, y_train)
    print(f"R² treinamento: {model.score(X_train, y_train):.3f}")
    print(f"R² teste: {model.score(X_test, y_test):.3f}")

### Logaritmo + Min-Max

Quando as colunas esperadas estão presentes, aplicamos `log10` em `Energia`, `Artot` e no alvo, seguido de normalização Min-Max nos atributos.

In [ ]:
if GAZOLA_PATH.exists():
    X_transformed = X.copy()
    for col in ["Energia", "Artot"]:
        if col in X_transformed.columns:
            X_transformed[col] = np.log10(X_transformed[col].clip(lower=1e-12))

    y_transformed = np.log10(y.clip(lower=1e-12))
    scaler = MinMaxScaler()
    X_scaled = scaler.fit_transform(X_transformed)

    X_train2, X_test2, y_train2, y_test2 = train_test_split(
        X_scaled, y_transformed, test_size=0.1, random_state=0
    )
    model2 = LinearRegression().fit(X_train2, y_train2)
    print(f"R² treinamento (transformado): {model2.score(X_train2, y_train2):.3f}")
    print(f"R² teste (transformado): {model2.score(X_test2, y_test2):.3f}")

## Conclusão

Este projeto demonstra que preparação de dados não é apenas limpeza: escolhas sobre valores ausentes, representação de categorias, engenharia de atributos, escala e distribuição podem alterar a forma como os dados são apresentados a um modelo.

> **Nota:** os resultados numéricos dependem dos arquivos de dados utilizados. O notebook não inclui os datasets para evitar redistribuição indevida.